In [ ]:
import json
import os.path
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import pickle
import scienceplots

In [ ]:
kitgreen="#009682"
green="#98BF64"
kitblue="#4664AA"
grey5="#f2f2f2ff"
grey30="#b3b3b3ff"

# British Data Overview and Preparation
# UK Data Overview and Preparation

## Geographic Data

In [ ]:
# Read the geographic boundary data of UK Local Authority Districts (LAD) (.shp file)
LAD_region = gpd.read_file(r"./dataset/HDRah-Data-PS-GB-1f63a32/Geo_data/LAD_DEC_2022_UK_BFC.shp", encoding='ISO-8859-1')
LAD_region = LAD_region.to_crs(epsg=4326)
LAD_region = LAD_region[["LAD22CD", "LAD22NM", "geometry"]]

In [ ]:
# Read the mapping data of UK Local Authority Districts (LAD) to Integrated Territorial Units for Statistics (ITL) regions
region_mapping = pd.read_csv(r"./dataset/Local_Authority_District_(April_2021)_to_LAU1_to_ITL3_to_ITL2_to_ITL1_(January_2021)_Lookup_in_United_Kingdom.csv")
region_mapping = region_mapping[["LAD21CD", "LAD21NM", "ITL321CD", "ITL221CD"]]

In [ ]:
# Add ITL (Integrated Territorial Units for Statistics) region codes via a mapping table
LAD_region["ITL3"] = LAD_region["LAD22CD"].apply(lambda x: region_mapping.loc[region_mapping["LAD21CD"]==x, "ITL321CD"].values[0])
LAD_region["ITL2"] = LAD_region["LAD22CD"].apply(lambda x: region_mapping.loc[region_mapping["LAD21CD"]==x, "ITL221CD"].values[0])
LAD_region.rename(columns={"LAD22CD": "LAD", "LAD22NM": "LADName"}, inplace=True)

In [ ]:
LAD_region

## Population and GVA Data

In [ ]:
# Land Use Distribution
# url: https://www.iea.org/countries/united-kingdom/efficiency-demand
uk_energy_consumption_share = {
    "transportation": 0.3410,
    "residential": 0.2792,
    "industrial": 0.1794,
    "commercial": 0.1397,
    "agricultural": 0.0105,
    "others": 0.0503
}

# GVA economic activity classification (SIC07 classification for GVA) from LLM
sic07_classification = {
    "industrial_gva": ["C (10-33)", "DE (35-39)", "F (41-43)"],
    "commercial_gva": ["G (45-47)","H (49-53)", "I (55-56)", "J (58-63)", "K (64-66)", "L (68)"],
    "agricultural_gva": ["AB (1-9)"],
    "others_gva": ["M (69-75)", "N (77-82)", "O (84)", "P (85)", "Q (86-88)", "R (90-93)", "S (94-96)", "T (97-98)"]
}

In [ ]:
# Loading population data
# url: https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/estimatesofthepopulationforenglandandwales
pop = pd.read_excel(r"./dataset/mye23tablesew.xlsx", sheet_name="MYE5", skiprows=7)
pop = pop[["Code", "Estimated Population mid-2023"]].set_index("Code")

In [ ]:
LAD_region = LAD_region.merge(pop, left_on='LAD', right_on='Code', how='left')
LAD_region.rename(columns={'Estimated Population mid-2023': 'population'}, inplace=True)

In [ ]:
nan_population_rows = LAD_region[LAD_region['population'].isnull()]
itl2_to_remove = nan_population_rows['ITL2'].unique()
print(f"Detected missing population data, will remove all related rows of the following ITL2 regions: {list(itl2_to_remove)}")
filtered_gdf = LAD_region[~LAD_region['ITL2'].isin(itl2_to_remove)]

In [ ]:
# 2a. Aggregate the attributes using pandas groupby
aggregation_functions = {'population': 'sum', 'ITL2': 'first'}
aggregated_attributes = filtered_gdf.drop(columns='geometry').groupby('ITL3').agg(aggregation_functions)
# 2b. Reset index to make 'ITL3' a column again
dissolved_geometry = filtered_gdf[['ITL3', 'geometry']].dissolve(by='ITL3')

In [ ]:
ITL3_region = dissolved_geometry.join(aggregated_attributes)

In [ ]:
total_england_population = ITL3_region.population.sum()
ITL3_region["residential_percent"] = ITL3_region["population"] / total_england_population

In [ ]:
ITL3_region

In [ ]:
# https://www.ons.gov.uk/economy/grossvalueaddedgva/datasets/nominalandrealregionalgrossvalueaddedbalancedbyindustry/current
gva_raw = pd.read_excel(r"./dataset/regionalgrossvalueaddedbalancedbyindustryandallitlregions.xlsx", sheet_name="Table 3c", skiprows=1)
gva_pivot = gva_raw.pivot_table(index='ITL code', columns='SIC07 code', values='2022', aggfunc='sum')

# Aggregate GVA by category
gva_by_category = pd.DataFrame(index=gva_pivot.index)
for category, codes in sic07_classification.items():
    # Ensure all codes exist in the columns to avoid KeyError
    valid_codes = [code for code in codes if code in gva_pivot.columns]
    gva_by_category[category] = gva_pivot[valid_codes].sum(axis=1)

In [ ]:
gva_by_category

In [ ]:
# total_gva = gva_by_category.sum().sum()
# gva_by_category = gva_by_category / total_gva
# gva_by_category

In [ ]:
gva_by_category["industrial_percent"] = gva_by_category["industrial_gva"]/ gva_by_category["industrial_gva"].sum()
gva_by_category["commercial_percent"] = gva_by_category["commercial_gva"]/ gva_by_category["commercial_gva"].sum()
gva_by_category["agricultural_percent"] = gva_by_category["agricultural_gva"]/ gva_by_category["agricultural_gva"].sum()
gva_by_category["others_percent"] = gva_by_category["others_gva"]/ gva_by_category["others_gva"].sum()

In [ ]:
ITL3_region = ITL3_region.merge(gva_by_category, left_on='ITL3', right_index=True, how='left')
ITL3_region.reset_index(inplace=True)

In [ ]:
# Calculate the area percentage of each LAD within its ITL region
ITL3_region = ITL3_region.to_crs(epsg=3857)
ITL3_region["area"] = ITL3_region.geometry.area
area_by_region = ITL3_region.groupby('ITL2')['area'].sum()
ITL3_region['area_percent'] = ITL3_region.apply(lambda row: (row['area'] / area_by_region[row['ITL2']]), axis=1)
ITL3_region = ITL3_region.to_crs(epsg=4326)

In [ ]:
ITL3_region

# Substation Data

In [ ]:
# Read UK substation data (including location, demand, capacity, etc.)
substations = gpd.read_file(r"./dataset/HDRah-Data-PS-GB-1f63a32/GB_PS_data_extend.csv")
substations["Geo(Long,Lat)"] = substations["Geo(Long,Lat)"].apply(lambda x: [float(coord) for coord in x.split(',')])
substations["geometry"] = substations.apply(lambda x: Point(x["Geo(Long,Lat)"][0], x["Geo(Long,Lat)"][1]), axis=1)
substations = substations.set_geometry("geometry")
substations = substations.set_crs(epsg=4326, inplace=True)

In [ ]:
# Aggregate substations at the same location (sum numeric values, take the first for others)
numeric_columns = substations.select_dtypes(include=np.number).columns.tolist()
non_numeric_columns = substations.select_dtypes(exclude=np.number).columns.tolist()

# Create aggregation dictionary: 'sum' for numeric columns, 'first' for non-numeric columns
agg_dict = {}
for col in numeric_columns:
    agg_dict[col] = 'sum'
for col in non_numeric_columns:
    agg_dict[col] = 'first'

del agg_dict["geometry"] # Geometry column is excluded from aggregation
# Group and aggregate
substations = substations.groupby("geometry", as_index=False).agg(agg_dict)
substations = gpd.GeoDataFrame(substations, geometry='geometry') # Convert back to GeoDataFrame

In [ ]:
# Data cleaning and formatting
substations["Demand (MVA)"] = pd.to_numeric(substations["Demand (MVA)"], errors='coerce')
substations["Firm Capacity (MVA)"] = pd.to_numeric(substations["Firm Capacity (MVA)"], errors='coerce')
substations["RegName"] = substations["RegName"].str.replace(" ", "")
substations = substations[["PS Name", "Demand (MVA)", "Firm Capacity (MVA)", "RegName", "geometry", "RegID"]]

In [ ]:
# Use spatial join to assign each substation to its corresponding LAD
substation_regions = gpd.sjoin(substations, ITL3_region, how="left", predicate="within")
substations["ITL3"] = substation_regions["ITL3"]
substations["ITL2"] = substation_regions["ITL2"]

In [ ]:
# Aggregate total demand and capacity of substations by itl3 region
region_stats = substation_regions.groupby(["ITL3"]).agg({"Demand (MVA)": "sum","Firm Capacity (MVA)": "sum"}).reset_index()
ITL3_region = ITL3_region.merge(region_stats, on=["ITL3"], how="left")

In [ ]:
ITL3_region.loc[ITL3_region['ITL2'].isin(["TLI3", "TLI4", "TLI5", "TLI6", "TLI7"]) , 'ITL2'] = "London"
substations.loc[substations['ITL2'].isin(["TLI3", "TLI4", "TLI5", "TLI6", "TLI7"]) , 'ITL2'] = "London"

In [ ]:
ITL3_region

In [ ]:
substations

In [ ]:
ITL3_region.to_file("./results/intermediate/ITL3_region.gpkg", driver='GPKG')
substations.to_file("./results/intermediate/substations.gpkg", driver='GPKG')

# Plotting the regions and substations

In [ ]:
interested_locations = {}
unique_itl2_regions = ITL3_region['ITL2'].unique()
for itl2_code in interested_locations.keys():
    interested_region = ITL3_region[ITL3_region['ITL2'] == itl2_code]
    interested_substations = substations[substations['ITL2'] == itl2_code]

    interested_locations[itl2_code] = {
        "region": interested_region,
        "substation": interested_substations
    }

In [ ]:
# for i, key in enumerate(interested_locations.keys()):
#     print(key)
#     fig, ax = plt.subplots(1, 1, figsize=(3.5, 2.5), dpi=300)  # Two-column width
#     interested_locations[key]["region"].plot(color=grey5, edgecolor=grey30, figsize=(7, 5), ax=ax, linewidth = 0.5, aspect='equal')
#     interested_locations[key]["substation"].plot(ax=ax, color=kitblue, markersize=0.2, aspect='equal')
#     # for x, y, label in zip(Ethos_regions.geometry.centroid.x, Ethos_regions.geometry.centroid.y, Ethos_regions['index']):
#     #     ax.text(x, y, label, fontsize=4, ha='center', va='center')
#     ax.tick_params(axis='x', labelsize=6)  # Adjust x-axis tick label size
#     ax.tick_params(axis='y', labelsize=6)
#     ax.set_xlabel(r'Longitude [$^\circ$E]', fontsize=8)
#     ax.set_ylabel(r'Latitude [$^\circ$N]', fontsize=8)
#     # ax.set_title('Original Surfaces and Points', fontsize=8)
#     plt.tight_layout()
#     # plt.savefig(f'./results/pictures/Figure_interested_region_{key}.eps', dpi=300, bbox_inches='tight')
#     plt.show()